# RAGAS Evaluation for Bhagavad Gita RAG

This notebook evaluates the RAG pipeline using [RAGAS](https://docs.ragas.io/) — a framework
for measuring retrieval-augmented generation quality without needing human annotations.

## Metrics
| Metric | What it measures |
|---|---|
| **Context Precision** | Are the retrieved chunks actually relevant to the question? |
| **Context Recall** | Does the retrieved context cover what the reference answer needs? |
| **Faithfulness** | Does the generated answer stay grounded in the retrieved context? |
| **Answer Relevancy** | Does the answer actually address the question asked? |

All metrics are scored 0–1 (higher is better).

## Requirements
- Ollama running locally with a model pulled (e.g. `ollama pull gemma3:12b`)
- The ChromaDB already indexed (`python -m src.ingest`)
- RAGAS and LangChain Ollama integration installed (cell below)

In [ ]:
# Install dependencies — run once, then restart the kernel if prompted.
# If pip is unavailable (uv-managed venv), install via terminal instead:
#   uv pip install ragas langchain-ollama langchain-community matplotlib
try:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install",
         "ragas", "langchain-ollama", "langchain-community", "matplotlib", "--quiet"],
        check=True,
    )
    print("Dependencies ready.")
except Exception:
    print("pip not available — install manually:\n"
          "  uv pip install ragas langchain-ollama langchain-community matplotlib")

## 1. Configuration

In [ ]:
import os, sys

PROJECT_ROOT = os.path.abspath(".")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

OLLAMA_MODEL    = "gemma3:12b"            # LLM for RAG answers
JUDGE_MODEL     = "gemma3:12b"            # LLM used by RAGAS to judge quality
OLLAMA_BASE_URL = "http://localhost:11434"

# ── External eval config (Crucible format) ───────────────────────────────────
# Point to a JSON file exported by the Crucible test-config app to use its
# questions instead of the built-in eval set.  Leave as None for the default.
# Expected keys: "question", "ground_truth", "answer", "contexts"
EVAL_CONFIG_PATH = None   # e.g. "crucible_agent_ragas_20260221_165018.json"

print(f"RAG model   : {OLLAMA_MODEL}")
print(f"Judge model : {JUDGE_MODEL}")
print(f"Eval config : {EVAL_CONFIG_PATH or 'built-in'}")

## 2. Evaluation Set

Questions and reference answers can come from **two sources**:

| Source | How to use |
|---|---|
| **Built-in** | Leave `EVAL_CONFIG_PATH = None` (default) |
| **Crucible JSON** | Set `EVAL_CONFIG_PATH` to the exported file path |

### Crucible JSON format
The Crucible test-config app exports files with this structure:

```json
{
  "question":     ["q1", "q2", ...],
  "ground_truth": ["expected answer 1", ""],
  "answer":       ["", ""],
  "contexts":     [[], []]
}
```

- **`ground_truth`** — used for *Context Recall*; leave empty if unknown
- **`answer` / `contexts`** — if already filled the RAG step is skipped; empty arrays mean "run the pipeline and fill in"

Reference answers are needed for **Context Recall**. 20–50 questions give stable scores.

In [ ]:
import json

# ── Built-in fallback eval set ────────────────────────────────────────────────
_BUILTIN = [
    {
        "question": "What does the Gita say about performing one's duty without attachment to results?",
        "reference": "The Gita teaches nishkama karma — acting without desire for the fruits of action. One should focus entirely on the quality of one's actions and let go of expectations about outcomes. This liberates a person from the cycle of attachment and disappointment.",
    },
    {
        "question": "How should I deal with grief and loss according to the Bhagavad Gita?",
        "reference": "The Gita says the soul is eternal and cannot be destroyed. What we grieve for is only the temporary physical form. True understanding of the self as imperishable brings equanimity in the face of loss. Grief arises from identifying with the body rather than the eternal self.",
    },
    {
        "question": "What is the meaning of dharma in the Bhagavad Gita?",
        "reference": "Dharma in the Gita refers to one's righteous duty — the role and responsibilities inherent to one's nature and station in life. It includes moral action, righteous conduct, and fulfilling one's obligations honestly. Arjuna's dharma as a warrior is to fight; abandoning it out of attachment would be adharma.",
    },
    {
        "question": "What does the Gita teach about the nature of the self (atman)?",
        "reference": "The Gita teaches that the true self (atman) is eternal, unborn, and indestructible. It cannot be cut by weapons, burned by fire, or dried by wind. The self is beyond the body and mind, and realising this nature is the foundation of spiritual liberation.",
    },
    {
        "question": "How can I stay calm and focused when everything around me is chaotic?",
        "reference": "The Gita teaches the practice of equanimity — remaining steady in pleasure and pain, success and failure. Through self-discipline, meditation, and detachment from outcomes, one can cultivate an inner stability that is unshaken by external circumstances.",
    },
    {
        "question": "What is the path of devotion (bhakti yoga) according to the Gita?",
        "reference": "Bhakti yoga is the path of loving devotion — dedicating all actions, thoughts, and feelings to the divine. The Gita says that a sincere devotee who remembers the divine at all times and surrenders the ego is very dear to the divine. Bhakti requires no special qualification, only sincere love and surrender.",
    },
    {
        "question": "How does the Gita view the cycle of birth and death?",
        "reference": "The Gita teaches that just as a person changes old garments for new ones, the soul discards old bodies and takes on new ones. Birth and death are transitions of the soul, not its end. The wise grieve neither for the living nor for the dead, understanding the soul's eternal nature.",
    },
    {
        "question": "What is the role of the mind in spiritual practice according to the Gita?",
        "reference": "The Gita describes the mind as both a friend and an enemy. When controlled, it is the best friend; when uncontrolled, it becomes the greatest enemy. Spiritual practice involves steadying the restless mind through discipline, non-attachment, and constant practice of inner focus.",
    },
    {
        "question": "What does the Gita say about the three gunas — sattva, rajas, and tamas?",
        "reference": "The three gunas are qualities of nature. Tamas is inertia and ignorance; rajas is passion and restlessness; sattva is clarity, harmony, and goodness. All beings are bound by these three qualities. Spiritual growth involves cultivating sattva and transcending all three gunas to reach a state of pure consciousness.",
    },
    {
        "question": "What advice does the Gita give to someone who feels their life has no purpose?",
        "reference": "The Gita says that every person has an inherent nature and a unique path of duty. Finding one's svadharma — one's own authentic path — is better than following another's. Purpose comes from understanding one's own nature, engaging with life fully, and offering one's actions as service without grasping at results.",
    },
]


def _load_crucible_json(path: str) -> list[dict]:
    """Load a Crucible-format JSON test config and return a normalised eval list.

    The JSON must have a "question" key.  Optional keys: "ground_truth",
    "answer", "contexts".  Empty strings/arrays mean "run the RAG pipeline".
    """
    with open(path) as f:
        cfg = json.load(f)
    questions     = cfg.get("question",     [])
    ground_truths = cfg.get("ground_truth", [""] * len(questions))
    answers       = cfg.get("answer",       [""] * len(questions))
    contexts      = cfg.get("contexts",     [[] for _ in questions])
    return [
        {
            "question":           q,
            "reference":          gt,
            "prefilled_answer":   a,
            "prefilled_contexts": ctx,
        }
        for q, gt, a, ctx in zip(questions, ground_truths, answers, contexts)
    ]


# ── Select source ─────────────────────────────────────────────────────────────
if EVAL_CONFIG_PATH:
    eval_items = _load_crucible_json(EVAL_CONFIG_PATH)
    print(f"Loaded {len(eval_items)} questions from: {EVAL_CONFIG_PATH}")
else:
    eval_items = [
        {"question": e["question"], "reference": e["reference"],
         "prefilled_answer": "", "prefilled_contexts": []}
        for e in _BUILTIN
    ]
    print(f"Using built-in eval set: {len(eval_items)} questions")

## 3. Run the RAG Pipeline on Each Question

In [ ]:
from src.rag import GitaRAG

rag = GitaRAG(model_provider="ollama", ollama_model=OLLAMA_MODEL)
records = []

for i, item in enumerate(eval_items, 1):
    print(f"[{i}/{len(eval_items)}] {item['question'][:70]}...")

    # If the config already has a filled answer and contexts, skip the RAG call.
    if item.get("prefilled_answer") and item.get("prefilled_contexts"):
        answer = item["prefilled_answer"]
        chunks = item["prefilled_contexts"]
    else:
        answer, _citations, context_str = rag.generate_answer(item["question"])
        chunks = [c.strip() for c in context_str.split("\n\n") if c.strip()]

    records.append({
        "user_input":         item["question"],
        "reference":          item["reference"],
        "response":           answer,
        "retrieved_contexts": chunks,
    })

print(f"\nDone. {len(records)} records collected.")

## 4. Score with RAGAS

RAGAS uses a judge LLM to score each sample on four metrics.

**Note on embeddings:** Chat models like `gemma3` do not support Ollama's `/api/embeddings`
endpoint (returns HTTP 501). We use `sentence-transformers` locally for the `AnswerRelevancy`
embedding scorer — no extra model to pull, runs fully offline.

In [ ]:
import os
from openai import OpenAI
from ragas import evaluate
from ragas.run_config import RunConfig
from ragas.llms import llm_factory
from ragas.embeddings import HuggingFaceEmbeddings as RagasHFEmbeddings
from ragas.dataset_schema import EvaluationDataset, SingleTurnSample
from ragas.metrics.collections import (
    ContextPrecision,
    ContextRecall,
    Faithfulness,
    AnswerRelevancy,
)

# Prevent HuggingFace network calls — model is already cached locally
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("HF_DATASETS_OFFLINE", "1")

# --- Judge LLM via Ollama's OpenAI-compatible endpoint ---
# RAGAS 0.4 dropped LangchainLLMWrapper; use llm_factory with an OpenAI client
# pointed at Ollama's /v1 endpoint (no real API key needed).
ollama_client = OpenAI(
    base_url=f"{OLLAMA_BASE_URL}/v1",
    api_key="ollama",          # Ollama ignores this value
)
judge_llm = llm_factory(model=JUDGE_MODEL, client=ollama_client)

# --- Embeddings: RAGAS-native HuggingFace (sentence-transformers, already installed) ---
# Note: parameter is `model=`, not `model_name=` in RAGAS 0.4
judge_emb = RagasHFEmbeddings(model="all-MiniLM-L6-v2")

# --- Metrics: PascalCase classes, must be instantiated with llm/embeddings ---
metrics = [
    ContextPrecision(llm=judge_llm),
    ContextRecall(llm=judge_llm),
    Faithfulness(llm=judge_llm),
    AnswerRelevancy(llm=judge_llm, embeddings=judge_emb),
]

# --- Dataset: RAGAS 0.4 EvaluationDataset with SingleTurnSample objects ---
samples = [
    SingleTurnSample(
        user_input=r["user_input"],
        reference=r["reference"],
        response=r["response"],
        retrieved_contexts=r["retrieved_contexts"],
    )
    for r in records
]
eval_dataset = EvaluationDataset(samples=samples)

print("Running RAGAS 0.4 evaluation (this may take several minutes)...")
results = evaluate(
    eval_dataset,
    metrics=metrics,
    run_config=RunConfig(timeout=180, max_retries=2, max_wait=60),
    raise_exceptions=False,
)
print(results)

## 5. View Results

In [10]:
import pandas as pd

df = results.to_pandas()

# Summary — one score per metric
metric_cols = [c for c in df.columns if c not in {"user_input", "reference", "response", "retrieved_contexts"}]
summary = df[metric_cols].mean().round(4).rename("score")

print("=== RAGAS Summary ===")
print(summary.to_string())

# Interpretation guide
print("""
Interpretation guide:
  > 0.8  — Good
  0.6–0.8 — Acceptable, room for improvement
  < 0.6  — Needs work

Low context_precision → retriever is pulling irrelevant chunks
Low context_recall    → retriever is missing important content
Low faithfulness      → model is hallucinating beyond the context
Low answer_relevancy  → answers are off-topic or too vague
""")

=== RAGAS Summary ===
context_precision   NaN
context_recall      NaN
faithfulness        NaN
answer_relevancy    NaN

Interpretation guide:
  > 0.8  — Good
  0.6–0.8 — Acceptable, room for improvement
  < 0.6  — Needs work

Low context_precision → retriever is pulling irrelevant chunks
Low context_recall    → retriever is missing important content
Low faithfulness      → model is hallucinating beyond the context
Low answer_relevancy  → answers are off-topic or too vague



In [11]:
# Per-question breakdown
display_cols = ["user_input"] + metric_cols
pd.set_option("display.max_colwidth", 60)
df[display_cols].sort_values("faithfulness", ascending=True)

,user_input,context_precision,context_recall,faithfulness,answer_relevancy
0,What does the Gita say about performing one's duty witho...,NaN,NaN,NaN,NaN
1,How should I deal with grief and loss according to the B...,NaN,NaN,NaN,NaN
2,What is the meaning of dharma in the Bhagavad Gita?,NaN,NaN,NaN,NaN
3,What does the Gita teach about the nature of the self (a...,NaN,NaN,NaN,NaN
4,How can I stay calm and focused when everything around m...,NaN,NaN,NaN,NaN
5,What is the path of devotion (bhakti yoga) according to ...,NaN,NaN,NaN,NaN
6,How does the Gita view the cycle of birth and death?,NaN,NaN,NaN,NaN
7,What is the role of the mind in spiritual practice accor...,NaN,NaN,NaN,NaN
8,"What does the Gita say about the three gunas — sattva, r...",NaN,NaN,NaN,NaN
9,What advice does the Gita give to someone who feels thei...,NaN,NaN,NaN,NaN


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = results.to_pandas()
metric_cols = [c for c in df.columns
               if c not in {"user_input", "reference", "response", "retrieved_contexts"}]

if df[metric_cols].isna().all().all():
    print("All metrics returned NaN — the judge LLM likely timed out or returned invalid JSON.")
    print("Tips:")
    print("  • Reduce eval set to 3–5 questions and re-run cells 3 and 4")
    print("  • Increase RunConfig(timeout=...) if Ollama is slow")
    print("  • Check Ollama is running: ollama ps")
else:
    colors = ["steelblue", "coral", "seagreen", "mediumpurple"]
    fig, axes = plt.subplots(1, len(metric_cols), figsize=(14, 4), sharey=True)

    for ax, col, color in zip(axes, metric_cols, colors):
        vals = df[col].dropna()
        ax.hist(vals, bins=10, range=(0, 1), color=color, edgecolor="white", linewidth=0.5)
        mean_val = vals.mean()
        ax.axvline(mean_val, color="black", linestyle="--", linewidth=1.5,
                   label=f"mean={mean_val:.2f}")
        ax.set_title(col.replace("_", " ").title())
        ax.set_xlim(0, 1)
        ax.legend(fontsize=8)

    plt.suptitle("RAGAS Metric Distributions", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig("ragas_results.png", dpi=150)
    plt.show()

    df.to_csv("ragas_results.csv", index=False)
    print("Saved: ragas_results.csv  ragas_results.png")